# Витрина транзакций ТЭ — 2. Клиент

AI-14134. Одна строка = один ИНН из факта. Таблица `sandbox_ai.tmp_shestopalov_te_client`. Сначала должна быть собрана `01_te_trx.ipynb`.

| колонка | источник |
|---|---|
| `okved` | основной ОКВЭД: `z_r2_ip_merchants` → `z_client` → `z_okved_period` (`c_main = 1`, действующий) → `z_okved`, как в `features` DAG техлида |
| `okopf` | сокращение формы (ООО, АО, ПАО): `z_cl_corp.c_forma` → `z_form_property.c_short_name`. У ИП — `ИП` |
| `region` | первые 2 цифры ОКАТО клиента `z_client.c_okato_code`: 45 — Москва, 40 — Санкт-Петербург |
| `filial` | филиал РСХБ, как `filial_rf` в `acq_06_cft_attrs.sql`: `z_r2_ip_merchants` → `z_depart` → `z_branch` |
| `ssp_ocrm` | ЗО клиента из OCRM, как в `acq_02_cdi_map.sql`. Обрезанные названия сводятся в ДМ / ДМСБ / ДКБ / ДРПА, как в `datamart_month_acquiring_v3.ipynb` |
| `ogrn` | `z_cl_corp.c_register_gos_reg_num_rec`, как в `acq_06_cft_attrs.sql` |
| `jur_address` | юридический адрес, `scd1_companies.c_jur_address` |
| `date_acquiring` | первое подключение: самая ранняя дата начала договора SA этого ИНН, включая закрытые |
| `date_agr_current` | начало текущего договора: самая ранняя дата начала договора SA из факта |

In [ ]:
from IPython.display import display
from rail_connectors.connection import connect

imp = connect(
    to="IMPALA",
    extra_options={"db": "sandbox_ai"},
    driver_args={"tez.queue.name": "ai"},
    kerberos={
        "keytab_path": "/home/jovyan/test_requests/tech.keytab",
        "use_credentials": True,
        "update_keytab": True,
    },
    user_params={"user_name": "Shestopalov-VYur"},
)
imp._init_connection()


In [ ]:
with imp:
    fact = imp.fetch("select count(*) as cnt from sandbox_ai.tmp_shestopalov_te_trx")
assert fact.iloc[0, 0] > 0, "Факт пуст: сначала выполните 01_te_trx.ipynb"

with imp:
    imp.execute("set MEM_LIMIT=8g")
    imp.execute("drop table if exists sandbox_ai.tmp_shestopalov_te_client")
    imp.execute("""
create table sandbox_ai.tmp_shestopalov_te_client
as
with agr as (
  select distinct
    f.inn,
    a.abs_agr_id,
    a.d_valid_from,
    c.c_jur_address
  from sandbox_ai.tmp_shestopalov_te_trx f
  join ods_alpha.scd1_agreements a
    on cast(a.n_agr as string) = f.n_agr
  left join ods_alpha.scd1_companies c
    on c.n_cmp = a.n_cmp_client
  where coalesce(a.ods_deleted_flg, '0') <> '1'
),
r2 as (
  select
    m.id as agr_id,
    corp.c_register_gos_reg_num_rec as ogrn,
    fp.c_short_name as okopf,
    substring(cl.c_okato_code, 1, 2) as region,
    br.c_shortlabel as filial,
    okv.c_code as okved
  from ods.scd1_z_r2_ip_merchants m
  left join ods.scd1_z_client cl
    on cl.id = m.c_cl_org
  left join ods.scd1_z_cl_corp corp
    on corp.id = m.c_cl_org
  left join ods.scd1_z_form_property fp
    on fp.id = corp.c_forma
  left join ods.scd1_z_depart dep
    on dep.id = m.c_depart
  left join ods.scd1_z_branch br
    on br.id = dep.c_filial
  left join ods.scd1_z_okved_period okvp
    on okvp.collection_id = cl.c_okved_in_period
   and okvp.c_main = '1'
   and okvp.c_date_end is null
  left join ods.scd1_z_okved okv
    on okv.id = okvp.c_okved
),
ocrm as (
  select
    regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') as inn,
    case
      when soe.x_area_resp like 'ДММБ%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ (ми%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ(ми%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ (ма%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ(ма%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ (ср%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ(ср%' then 'ДМСБ'
      when soe.x_area_resp like 'ДСБ%' then 'ДМСБ'
      when soe.x_area_resp like 'ДКБ%' then 'ДКБ'
      when soe.x_area_resp like 'ДРПА%' then 'ДРПА'
      when soe.x_area_resp like 'ДРА%' then 'ДРПА'
      when lower(soe.x_area_resp) like 'не под%' then 'Не подлежит сегментации'
      else null
    end as ssp_ocrm,
    row_number() over (
      partition by regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '')
      order by cast(soe.created as timestamp) desc, cast(soe.row_id as string) desc
    ) as rn
  from ocrm_ul.s_org_ext soe
  where regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') in (
      select inn from sandbox_ai.tmp_shestopalov_te_trx
    )
    and coalesce(soe.x_removed_flg, 'N') = 'N'
    and coalesce(soe.x_duplicate_flg, 'N') = 'N'
),
first_sa as (
  select
    regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn,
    min(a.d_valid_from) as date_acquiring
  from ods_alpha.scd1_agreements a
  join ods_alpha.scd1_companies c
    on c.n_cmp = a.n_cmp_client
  where upper(trim(cast(a.acq_class as string))) = 'SA'
    and coalesce(a.ods_deleted_flg, '0') <> '1'
    and regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') in (
      select inn from sandbox_ai.tmp_shestopalov_te_trx
    )
  group by regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '')
)
select
  agr.inn,
  max(r2.okved) as okved,
  coalesce(max(r2.okopf), case when length(agr.inn) = 12 then 'ИП' end) as okopf,
  max(r2.region) as region,
  max(r2.filial) as filial,
  max(o.ssp_ocrm) as ssp_ocrm,
  max(r2.ogrn) as ogrn,
  max(agr.c_jur_address) as jur_address,
  max(fs.date_acquiring) as date_acquiring,
  min(agr.d_valid_from) as date_agr_current
from agr
left join r2
  on r2.agr_id = agr.abs_agr_id
left join ocrm o
  on o.inn = agr.inn
 and o.rn = 1
left join first_sa fs
  on fs.inn = agr.inn
group by agr.inn
""")
print("собрано sandbox_ai.tmp_shestopalov_te_client")

In [ ]:
with imp:
    display(imp.fetch("""
select
  count(*) as rows_cnt,
  count(distinct inn) as inn_cnt,
  sum(case when okved is null then 1 else 0 end) as okved_null,
  sum(case when okopf is null then 1 else 0 end) as okopf_null,
  sum(case when region is null then 1 else 0 end) as region_null,
  sum(case when filial is null then 1 else 0 end) as filial_null,
  sum(case when ssp_ocrm is null then 1 else 0 end) as ssp_ocrm_null,
  sum(case when ogrn is null then 1 else 0 end) as ogrn_null,
  sum(case when jur_address is null then 1 else 0 end) as jur_address_null,
  sum(case when date_acquiring is null then 1 else 0 end) as date_acquiring_null,
  sum(case when date_agr_current is null then 1 else 0 end) as date_agr_current_null
from sandbox_ai.tmp_shestopalov_te_client
"""))
    display(imp.fetch("""
select okopf, count(*) as inn_cnt
from sandbox_ai.tmp_shestopalov_te_client
group by okopf
order by inn_cnt desc
"""))
    display(imp.fetch("""
select length(inn) as inn_len, count(*) as okopf_null
from sandbox_ai.tmp_shestopalov_te_client
where okopf is null
group by length(inn)
"""))
    display(imp.fetch("""
select ssp_ocrm, count(*) as inn_cnt
from sandbox_ai.tmp_shestopalov_te_client
group by ssp_ocrm
order by inn_cnt desc
"""))
    display(imp.fetch("""
select region, filial, count(*) as inn_cnt
from sandbox_ai.tmp_shestopalov_te_client
group by region, filial
order by inn_cnt desc
limit 30
"""))
    display(imp.fetch("""
select
  year(date_acquiring) as acq_year,
  count(*) as inn_cnt,
  sum(case when date_agr_current > date_acquiring then 1 else 0 end) as new_agr_later
from sandbox_ai.tmp_shestopalov_te_client
group by year(date_acquiring)
order by acq_year
"""))
    display(imp.fetch("select * from sandbox_ai.tmp_shestopalov_te_client limit 20"))